# Ejecutar AlphaSymbolic

Ejemplo de regresión simbólica con el motor GPU de AlphaSymbolic/WarpSymbolic y progreso en vivo. En Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** y ejecuta las celdas en orden.

## Preparación

Esta celda descarga el repositorio e instala sus dependencias en Colab. Los cambios locales del código deben estar publicados en GitHub para que Colab los descargue.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path('/content/Algoritmo-Genetico---Formulas')
if not (repo / 'pyproject.toml').is_file():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/juansito17/Algoritmo-Genetico---Formulas.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo), 'matplotlib'], check=True)
help_text = subprocess.run([sys.executable, '-m', 'warpsymbolic.cli.run_gpu_console', '--help'], cwd=repo, capture_output=True, text=True, check=True).stdout
if '--progress-interval' not in help_text:
    raise RuntimeError('La versión descargada aún no incluye el progreso en vivo. Publica los cambios del repositorio en GitHub y reinicia esta sesión.')
print('Proyecto instalado desde:', repo)

## Compilar el motor CUDA

La primera compilación puede tardar varios minutos. Se compila para la arquitectura de la GPU asignada; el archivo compilado se reutiliza mientras dure la sesión de Colab.

In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Activa una GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución.')
major, minor = torch.cuda.get_device_capability()
build_env = os.environ.copy()
build_env['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
build_env['MAX_JOBS'] = '2'
cuda_dir = repo / 'src' / 'warpsymbolic' / 'gpu' / 'cuda'
if not list(cuda_dir.glob('rpn_cuda_native*.so')):
    subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], cwd=cuda_dir, env=build_env, check=True)
from warpsymbolic.gpu.cuda_loader import load_rpn_cuda_native
print('Extensión CUDA:', load_rpn_cuda_native().__file__)

In [ ]:
import numpy as np
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Activa GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución.')
DEVICE = 'cuda'
gpu = torch.cuda.get_device_properties(0)
memory_gb = gpu.total_memory / 2**30
if 'A100' not in gpu.name:
    raise RuntimeError(f'Se esperaba una A100; Colab asignó {gpu.name}. Ajusta la configuración antes de continuar.')
print(f'GPU: {gpu.name} | VRAM: {memory_gb:.1f} GiB')

## Datos de ejemplo

Sustituye `X` e `y` por tus datos: `X` debe tener forma `(muestras, variables)` e `y` forma `(muestras,)`.

In [ ]:
rng = np.random.default_rng(42)
X = rng.uniform(-3, 3, size=(1024, 2))
y = np.sin(2 * X[:, 0]) + 0.5 * X[:, 1] ** 2 + X[:, 0] * X[:, 1]
print('X:', X.shape, '| y:', y.shape)

In [ ]:
import pandas as pd

# El evaluador usa la ruta CUDA fusionada hasta 1_500_000 individuos.
csv_path = Path('/content/alphasymbolic_datos.csv')
pd.DataFrame({'x0': X[:, 0], 'x1': X[:, 1], 'y': y}).to_csv(csv_path, index=False)
command = [
    sys.executable, '-u', '-m', 'warpsymbolic.cli.run_gpu_console',
    str(csv_path), '--target', 'y', '--device', 'cuda', '--legacy',
    '--pop-size', '1500000', '--islands', '20',
    '--max-time', '3600',
    '--progress-interval', '10',
]
print('Ejecutando:', ' '.join(command), flush=True)
subprocess.run(command, cwd=repo, check=True)